# Installed Base Reliability and Service Cost Forecasting
## Part 4: Forecasting a New Generation with Sparse Data

### The problem
Parts 2 and 3 had hundreds of failures per component to learn from. A **new tool generation** has almost none. At KLA, a new inspection platform for a new process node ships to fabs with little or no field history, yet service contracts for it must be priced and reserved on day one.

That leaves three options, and two of them are bad:

| Approach | What it does with a new platform | Failure mode |
|---|---|---|
| Platform only estimate | failures ÷ exposure on the new platform alone | 0 failures in the first month means "never fails"; 1 failure in a short window means "fails constantly" |
| Fleet average | assumes the new platform behaves like the average of old ones | never learns that this platform is different, no matter how much data arrives |
| **Hierarchical Bayesian** | starts from what old platforms imply, moves toward the new platform's own data as it accumulates | needs a model of how different platforms tend to be, which is exactly what the hierarchy estimates |

### The core idea in one numerical example
A new platform logs **3 failures in 800 tool months**. The platform only estimate is 3 / 800 = **0.375%** per tool month.

Older platforms suggest a rate near 1%, and we encode that as a Gamma prior with shape 5 and rate 500 (mean 5 / 500 = 1%). The prior behaves like **500 tool months of pseudo experience** containing 5 failures. Because the Gamma prior and Poisson failure counts are conjugate, the update is just addition:

$$\text{Gamma}(5,\ 500) \;+\; (3 \text{ failures},\ 800 \text{ tool months}) \;\rightarrow\; \text{Gamma}(5 + 3,\ 500 + 800) = \text{Gamma}(8,\ 1300)$$

Posterior mean = 8 / 1300 = **0.615%**. Equivalently, a weighted average: the platform's own data gets weight 800 / (800 + 500) = 0.615, the prior gets the rest, so 0.615 × 0.375% + 0.385 × 1% = 0.615%.

**What the gap costs.** For 2,000 tools at $18,000 per failure:
1. The platform only estimate implies 2,000 × 0.375% × $18,000 = **$135,000 per month**.
2. The Bayesian estimate implies 2,000 × 0.615% × $18,000 = **$221,400 per month**.
3. If the Bayesian view is closer to the truth, pricing on the raw rate under reserves by $86,400 a month, or about **$3.1M over a three year contract**.

### Where does the prior come from? The hierarchy
The example assumed the prior Gamma(5, 500). In practice both numbers are **learned from the existing platforms**:
1. Their **average** rate sets the prior mean.
2. How much they **differ from each other** sets the prior's strength. If existing platforms all behave alike, a new one probably will too, so the prior is strong and shrinks hard. If they differ wildly, the prior is weak and the new platform's own data takes over quickly.

That second point is what makes the model hierarchical: the amount of borrowing is estimated, not chosen.

### The experiment
Pretend each platform in turn is brand new. Hide it, learn the hierarchy from the other three, then give it a launch date and reveal its data as it would have arrived: 15, 30, 60, 90, and 180 days after launch, repeated for three launch dates. At each point, forecast the failures it will have over the rest of the year with each of the three approaches, and compare with what actually happened. The question a service planner cares about: **how many weeks of field data until the forecast is trustworthy, and which method gets there first?**

### Project roadmap
| Part | Notebook | Output |
|---|---|---|
| 1 | Data foundation and failure behavior | Clean component lifetime table with censoring |
| 2 | Survival modeling | Weibull parameters per component, validated hazard ratios |
| 3 | Fleet cost forecast | Backtested quarterly failure and cost forecast |
| **4** | **New generation forecasting (this notebook)** | **Hierarchical Bayesian platform rates, tested on held out platforms** |

## 1. Setup
Everything is implemented with NumPy and SciPy. Because the model is conjugate, the posterior can be computed exactly on a grid, with no sampler to tune or diagnose.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import norm

DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")

COMPONENTS = ["comp1", "comp2", "comp3", "comp4"]
COLORS = {"comp1": "#534AB7", "comp2": "#D85A30", "comp3": "#1D9E75", "comp4": "#BA7517"}
METHOD_COLORS = {"platform_only": "#D85A30", "fleet_average": "grey", "hierarchical": "#534AB7"}
RNG = np.random.default_rng(11)

AIC_THRESHOLD = 2
N_DRAWS = 20_000

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

## 2. Load the lifetimes and reuse the Part 2 and 3 tools
`fit_best` (2 or 3 parameter Weibull chosen by AIC, the Akaike Information Criterion, with Part 2's safeguard against degenerate 3 parameter fits) and `history_as_of` (rebuild the data as it looked on a given date) are the same functions as in Part 3. `window_view` keeps only what happened between two dates, which the experiment uses to give the pretend new platform a launch date.

In [ ]:
lifetimes = pd.read_csv(DATA_PROCESSED / "component_lifetimes.csv", parse_dates=["start", "end"])
bounds = pd.read_csv(DATA_RAW / "PdM_telemetry.csv", usecols=["datetime"], parse_dates=["datetime"])
OBS_START, OBS_END = bounds["datetime"].min(), bounds["datetime"].max()
del bounds

lifetimes["entry"] = ((OBS_START - lifetimes["start"]).dt.total_seconds() / 86_400).clip(lower=0)
lifetimes = lifetimes[lifetimes["duration_days"] > lifetimes["entry"]].reset_index(drop=True)
PLATFORMS = sorted(lifetimes["model"].unique())


def _unpack(x, gamma_max):
    beta, eta = np.exp(x[0]), np.exp(x[1])
    gamma = gamma_max / (1 + np.exp(-x[2])) if len(x) == 3 else 0.0
    return beta, eta, gamma


def weibull_negloglik(x, t, event, entry, gamma_max):
    beta, eta, gamma = _unpack(x, gamma_max)
    z = np.clip(t - gamma, 1e-12, None) / eta
    z_entry = np.clip(entry - gamma, 0, None) / eta
    log_h = np.log(beta / eta) + (beta - 1) * np.log(z)
    return -(np.sum(event * log_h) - np.sum(z ** beta) + np.sum(z_entry ** beta))


def fit_weibull(t, event, entry, location=False) -> dict:
    t, event, entry = (np.asarray(a, float) for a in (t, event, entry))
    gamma_max = 0.99 * t[event == 1].min()
    x0 = [0.0, np.log(t.mean())] + ([0.0] if location else [])
    res = minimize(weibull_negloglik, x0, args=(t, event, entry, gamma_max), method="Nelder-Mead",
                   options={"maxiter": 5_000, "xatol": 1e-6, "fatol": 1e-8})
    if not res.success:
        raise RuntimeError(res.message)
    beta, eta, gamma = _unpack(res.x, gamma_max)
    return {"beta": beta, "eta": eta, "gamma": gamma, "gamma_max": gamma_max, "AIC": 2 * len(x0) + 2 * res.fun}


def fit_best(t, event, entry) -> dict:
    f2 = fit_weibull(t, event, entry, location=False)
    try:
        f3 = fit_weibull(t, event, entry, location=True)
    except RuntimeError:
        return f2
    valid_3p = f3["beta"] >= 1 and f3["gamma"] < 0.95 * f3["gamma_max"]
    return f3 if (f2["AIC"] - f3["AIC"] > AIC_THRESHOLD and valid_3p) else f2


def history_as_of(lt: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    d = lt[lt["start"] < origin].copy()
    running = d["end"] > origin
    d.loc[running, "end"] = origin
    d.loc[running, "event"] = 0
    d["duration_days"] = (d["end"] - d["start"]).dt.total_seconds() / 86_400
    return d[d["duration_days"] > d["entry"]].reset_index(drop=True)


def window_view(lt: pd.DataFrame, t0: pd.Timestamp, t1: pd.Timestamp) -> pd.DataFrame:
    # Only what happened between t0 and t1: parts are watched from t0 (left truncation) and cut at t1
    d = lt[(lt["end"] > t0) & (lt["start"] < t1)].copy()
    running = d["end"] > t1
    d.loc[running, "end"] = t1
    d.loc[running, "event"] = 0
    d["duration_days"] = (d["end"] - d["start"]).dt.total_seconds() / 86_400
    d["entry"] = np.maximum(d["entry"], (t0 - d["start"]).dt.total_seconds() / 86_400)
    return d[d["duration_days"] > d["entry"]].reset_index(drop=True)


pd.crosstab(lifetimes["comp"], lifetimes["model"], values=lifetimes["event"], aggfunc="sum").fillna(0).astype(int)

The table above is the raw material: failures per component and platform. Every cell is a group the hierarchy will reason about.

## 3. Measuring exposure in units of risk
A platform's failure rate needs a denominator. Plain tool days would be wrong here: Part 2 showed that every component **wears out**, so a day at age 150 carries far more risk than a day at age 10. Two platforms with the same tool days but different part ages would look different for no real reason.

The fix is to measure exposure with the **cumulative hazard** of the component's Weibull baseline:

$$H(t) = \left(\frac{\max(t - \gamma,\ 0)}{\eta}\right)^{\beta}, \qquad \text{exposure of one lifetime} = H(\text{duration}) - H(\text{entry})$$

### Numerical example
With β = 2 and η = 150 days (γ = 0), a part observed from age 0 to 50 contributes H(50) = (50/150)² = **0.111** units of risk. A part observed from age 100 to 150, the same 50 days, contributes H(150) − H(100) = 1 − 0.444 = **0.556**, five times more. A platform whose parts happened to be older should have more failures, and this denominator expects them.

### What the rate means
With exposure measured this way, a platform's rate λ is a **hazard multiplier** relative to the component baseline: λ = 1 means "fails exactly as the baseline predicts", λ = 1.5 means 50% more failures than expected given the ages observed. This is the same proportional hazards idea as Cox in Part 2, and it keeps the Poisson likelihood exact: for a platform with y failures and exposure E, the likelihood of λ is proportional to λ^y · e^(−λE).

In [ ]:
def cumulative_hazard(t, p: dict) -> np.ndarray:
    return (np.clip(np.asarray(t, float) - p["gamma"], 0, None) / p["eta"]) ** p["beta"]


def fit_baselines(lt: pd.DataFrame) -> dict:
    return {c: fit_best(g["duration_days"], g["event"], g["entry"]) for c, g in lt.groupby("comp")}


def cell_stats(lt: pd.DataFrame, baselines: dict) -> pd.DataFrame:
    rows = []
    for (comp, model), g in lt.groupby(["comp", "model"]):
        p = baselines[comp]
        exposure = (cumulative_hazard(g["duration_days"], p) - cumulative_hazard(g["entry"], p)).sum()
        rows.append({"comp": comp, "model": model, "failures": int(g["event"].sum()), "exposure": exposure})
    cells = pd.DataFrame(rows)
    cells["raw_rate"] = cells["failures"] / cells["exposure"]
    return cells


all_cells = cell_stats(lifetimes, fit_baselines(lifetimes))
all_cells.round(3)

With baselines fitted on the whole fleet, `raw_rate` hovers around 1 on average, and the spread around 1 is the platform to platform variation the hierarchy has to learn. A raw rate of 0 means a platform never failed that component.

## 4. The hierarchical model

### Structure
For component c on platform p, with y failures and exposure E:

$$y_{c,p} \sim \text{Poisson}(\lambda_{c,p} \, E_{c,p}), \qquad \lambda_{c,p} \sim \text{Gamma}\!\left(\alpha,\ \frac{\alpha}{\mu}\right)$$

1. **μ** is the typical multiplier across platforms, and the Gamma mean.
2. **α** controls how alike platforms are. The Gamma's standard deviation is μ / √α, so α = 25 means platforms typically sit within ±20% of μ, while α = 1 means they can easily differ by 100%.
3. α / μ is the prior's **pseudo exposure**, the number that played the role of 500 tool months in the introduction. Large α means strong shrinkage.

Because every multiplier is relative to its own component baseline, all 16 component and platform cells share one (μ, α). That pools evidence across components, which matters when only three platforms remain after holding one out.

### Hyperpriors
1. log μ ~ Normal(0, 1): centered on 1, since the baselines already describe the average platform.
2. log α ~ Normal(log 5, 2): weak, spanning from near identical platforms to wildly different ones. Section 8 checks how much this choice matters.

### Exact posterior on a grid
Integrating λ out of the Poisson and Gamma gives a **negative binomial** for each cell's count, so the likelihood of (μ, α) has a closed form:

$$\log p(y \mid \mu, \alpha) = \log\Gamma(y + \alpha) - \log\Gamma(\alpha) - \log y! + \alpha \log\frac{\alpha}{\alpha + \mu E} + y \log\frac{\mu E}{\alpha + \mu E}$$

We evaluate it on a 100 × 100 grid of (μ, α), multiply by the hyperprior, and normalize. For a new platform with y failures and exposure E, the posterior of its multiplier is then a weighted mix of conjugate updates, one per grid point:

$$\lambda_{\text{new}} \mid \text{data} \;\sim\; \sum_k w_k \,\text{Gamma}\!\left(\alpha_k + y,\ \frac{\alpha_k}{\mu_k} + E\right)$$

It is the introduction's example, repeated for every plausible prior and weighted by how well each prior explains the existing platforms.

In [ ]:
MU_GRID = np.exp(np.linspace(np.log(0.05), np.log(20), 100))
ALPHA_GRID = np.exp(np.linspace(np.log(0.05), np.log(500), 100))
MU, ALPHA = np.meshgrid(MU_GRID, ALPHA_GRID, indexing="ij")


def log_prior(prior: str = "default") -> np.ndarray:
    lp_mu = norm.logpdf(np.log(MU), 0, 1)
    if prior == "default":
        lp_alpha = norm.logpdf(np.log(ALPHA), np.log(5), 2)
    elif prior == "flat_log":
        lp_alpha = np.zeros_like(ALPHA)
    elif prior == "expects_different":
        lp_alpha = norm.logpdf(np.log(ALPHA), np.log(1), 1)
    else:
        raise ValueError(prior)
    return lp_mu + lp_alpha


def hyper_posterior(cells: pd.DataFrame, prior: str = "default") -> np.ndarray:
    lp = log_prior(prior)
    for y, E in zip(cells["failures"], cells["exposure"]):
        mE = MU * E
        lp = lp + (gammaln(y + ALPHA) - gammaln(ALPHA) - gammaln(y + 1)
                   + ALPHA * np.log(ALPHA / (ALPHA + mE)) + y * np.log(mE / (ALPHA + mE)))
    w = np.exp(lp - lp.max())
    return w / w.sum()


def cell_posterior(y: float, E: float, w: np.ndarray, n_draws: int = N_DRAWS, mu=None, alpha=None) -> dict:
    mu = MU if mu is None else np.asarray(mu, float)
    alpha = ALPHA if alpha is None else np.asarray(alpha, float)
    a = alpha.ravel() + y
    b = alpha.ravel() / mu.ravel() + E
    mean = float(np.sum(w.ravel() * a / b))
    k = RNG.choice(w.size, size=n_draws, p=w.ravel())
    draws = RNG.gamma(a[k], 1 / b[k])
    return {"mean": mean, "p10": float(np.percentile(draws, 10)), "p90": float(np.percentile(draws, 90)), "draws": draws}


# Verify 1: a single prior point (mean 1%, shape 5, so rate 500) reproduces the introduction's update
got = cell_posterior(3, 800, np.array([1.0]), mu=[0.01], alpha=[5.0])
print(f"Conjugate check: Gamma(5, 500) + 3 failures in 800 tool months -> mean {got['mean']:.5f} "
      f"(hand calculation 8 / 1300 = {8 / 1300:.5f})")
assert abs(got["mean"] - 8 / 1300) < 1e-12

# Verify 2: simulate 16 cells from known mu = 1.0, alpha = 8 and check the hyperposterior recovers them
true_mu, true_alpha = 1.0, 8.0
sim = pd.DataFrame({"exposure": RNG.uniform(20, 80, 16)})
sim["failures"] = RNG.poisson(RNG.gamma(true_alpha, true_mu / true_alpha, 16) * sim["exposure"])
w_sim = hyper_posterior(sim)
mu_hat, alpha_hat = np.sum(w_sim * MU), np.exp(np.sum(w_sim * np.log(ALPHA)))
print(f"Simulated check: true mu {true_mu}, alpha {true_alpha}  |  posterior mean mu {mu_hat:.2f}, "
      f"geometric mean alpha {alpha_hat:.1f}")
assert 0.8 < mu_hat < 1.25 and 2 < alpha_hat < 40

The first check confirms the posterior function performs exactly the conjugate update from the introduction. The second confirms the hierarchy recovers a known population: μ tightly, α within the wide band that 16 cells can support. α is always the harder parameter; it is estimated from how the groups differ, and there are only 16 groups.

## 5. The fleet view: shrinkage across all platforms
Before the experiment, fit the hierarchy on every platform to see what it learns. The shrinkage plot shows each cell's raw rate and where the model moves it. Cells with little exposure move a lot toward the center; cells with plenty of exposure barely move.

In [ ]:
w_all = hyper_posterior(all_cells)
post = [cell_posterior(y, E, w_all) for y, E in zip(all_cells["failures"], all_cells["exposure"])]
all_cells["posterior_mean"] = [p["mean"] for p in post]
all_cells["p10"] = [p["p10"] for p in post]
all_cells["p90"] = [p["p90"] for p in post]

mu_mean = np.sum(w_all * MU)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
for comp in COMPONENTS:
    sub = all_cells[all_cells["comp"] == comp]
    for _, r in sub.iterrows():
        ax.plot([0, 1], [r["raw_rate"], r["posterior_mean"]], "-o", color=COLORS[comp], alpha=0.8)
    ax.plot([], [], "-o", color=COLORS[comp], label=comp)
ax.axhline(mu_mean, color="black", ls=":", label=f"learned center μ ≈ {mu_mean:.2f}")
ax.set_xticks([0, 1], ["raw rate", "hierarchical posterior"])
ax.set(title="Shrinkage: every cell moves toward the center", ylabel="Hazard multiplier")
ax.legend(fontsize=8)

ax = axes[1]
marg_alpha = w_all.sum(axis=0)
ax.plot(ALPHA_GRID, marg_alpha, color="#534AB7")
ax.set(xscale="log", title="How alike are platforms? Posterior of α", xlabel="α (larger means more alike)",
       ylabel="Posterior probability")
plt.tight_layout()
plt.show()

all_cells.round(3)

**Reading the α posterior.** If most of the mass sits at large α, the existing platforms are similar and a new platform will be pulled hard toward the fleet. If it sits at small α (for example because one platform never fails a component that others fail often), the model has learned that platforms can be very different and will trust a new platform's own data sooner. Either way the data decides, not the analyst.

## 6. The experiment: pretend each platform is new
For each held out platform:
1. **Learn from the others only.** Refit the component Weibull baselines and the hierarchy on the other three platforms, so nothing about the "new" platform leaks into the prior.
2. **Give it a launch date and reveal its data over time.** The platform "launches" on day 30, 60, or 90 of the data (each launch is a separate run). Its history is rebuilt 15, 30, 60, 90, and 180 days after launch, counting only failures and exposure since launch.
3. **Estimate its multiplier three ways:** platform only (y / E), fleet average (the learned center μ), and the hierarchical posterior.
4. **Forecast the rest of the year and score it.** Predicted remaining failures = estimated multiplier × remaining exposure. Compare with the failures that actually happened after the cutoff.

Scoring on **future failures** rather than on a "true" multiplier has two advantages. It is a genuine out of sample forecast in time, exactly the situation of a planner at day 30. And cells with tiny exposure, where any rate estimate is wild, contribute only the few failures they actually produce instead of dominating the average.

**Why several launch dates, and why not day 0?** Failures rise and fall across the calendar for the whole fleet at once (Part 1's monthly chart shows quiet and busy months). A short window after launch mostly measures which month it fell in, not the platform, and a single launch date would let that luck decide the result. Averaging over three launch dates spreads the test across different parts of the year. Day 0 is excluded because the start of the failure log is not typical, as the check below shows: the first week has far fewer failures than a normal week, right after the batch stamped on the log's first timestamp that Part 1 quarantined.

One simplification to state: the remaining exposure is taken from what the parts actually went on to run. In production it would be projected from the installed base, as in Part 3. Conditioning on it here isolates the question this notebook asks: **how good is the rate estimate?**

The hierarchical forecast also gets an 80% predictive interval: draw a multiplier from the posterior, draw a Poisson count with that rate over the remaining exposure, repeat.

In [ ]:
weekly = lifetimes.loc[lifetimes["event"] == 1, "end"].dt.to_period("W").value_counts().sort_index()
print(f"Failures in the first week of the log: {weekly.iloc[0]}  |  median week: {weekly.median():.0f}")

LAUNCH_DAYS = [30, 60, 90]
CUTOFF_DAYS = [15, 30, 60, 90, 180]
assert max(LAUNCH_DAYS) + max(CUTOFF_DAYS) < (OBS_END - OBS_START).days, "Last cutoff must leave some year to forecast"

rows = []
for held_out, launch_day in [(p, d) for p in PLATFORMS for d in LAUNCH_DAYS]:
    LAUNCH = OBS_START + pd.Timedelta(days=launch_day)
    known = lifetimes[lifetimes["model"] != held_out]
    new = lifetimes[lifetimes["model"] == held_out]

    baselines = fit_baselines(known)
    w = hyper_posterior(cell_stats(known, baselines))
    center = float(np.sum(w * MU))
    full = cell_stats(window_view(new, LAUNCH, OBS_END), baselines).set_index("comp")

    for days in CUTOFF_DAYS:
        seen = cell_stats(window_view(new, LAUNCH, LAUNCH + pd.Timedelta(days=days)), baselines).set_index("comp")
        for comp in COMPONENTS:
            y = seen.loc[comp, "failures"] if comp in seen.index else 0
            E = seen.loc[comp, "exposure"] if comp in seen.index else 0.0
            y_rem = full.loc[comp, "failures"] - y
            E_rem = max(full.loc[comp, "exposure"] - E, 0.0)
            p = cell_posterior(y, E, w)
            pred_counts = RNG.poisson(p["draws"] * E_rem)
            rates = {"platform_only": y / E if E > 0 else np.nan, "fleet_average": center, "hierarchical": p["mean"]}
            rows.append({"held_out": held_out, "launch_day": launch_day, "comp": comp, "days": days,
                         "failures_seen": y, "exposure_seen": E, "exposure_remaining": E_rem,
                         "actual_remaining": y_rem, "full_year_rate": full.loc[comp, "raw_rate"],
                         **{f"rate_{m}": v for m, v in rates.items()},
                         **{f"pred_{m}": v * E_rem for m, v in rates.items()},
                         "hier_rate_p10": p["p10"], "hier_rate_p90": p["p90"],
                         "pred_p10": np.percentile(pred_counts, 10), "pred_p90": np.percentile(pred_counts, 90)})

experiment = pd.DataFrame(rows)
experiment["covered"] = experiment["actual_remaining"].between(experiment["pred_p10"], experiment["pred_p90"])
experiment[experiment["days"] == 30].round(2)

### Scoring: forecast error on the rest of the year, by days of data
For each cutoff, over all held out platforms and components:
1. **MAE (Mean Absolute Error):** typical miss in remaining failures per component per platform.
2. **WAPE (Weighted Absolute Percentage Error):** total absolute miss ÷ total actual remaining failures.
3. **Bias:** average of forecast − actual. Negative means under forecasting, which becomes under reserving.
4. **Coverage** of the hierarchical 80% predictive interval.

A platform only estimate with no exposure at all is undefined; those rows count as missing for that method.

In [ ]:
METHODS = ["platform_only", "fleet_average", "hierarchical"]


def score_by_days(df: pd.DataFrame) -> pd.DataFrame:
    out = {}
    for days, g in df.groupby("days"):
        row = {}
        for m in METHODS:
            err = g[f"pred_{m}"] - g["actual_remaining"]
            row[f"MAE_{m}"] = err.abs().mean()
            row[f"WAPE_{m}"] = err.abs().sum() / g["actual_remaining"].sum()
            row[f"bias_{m}"] = err.mean()
        row["hier_80pct_coverage"] = g["covered"].mean()
        out[days] = row
    s = pd.DataFrame(out).T
    s.index.name = "days"
    s["hier_vs_platform_only"] = 1 - s["MAE_hierarchical"] / s["MAE_platform_only"]
    s["hier_vs_fleet_average"] = 1 - s["MAE_hierarchical"] / s["MAE_fleet_average"]
    return s


scores = score_by_days(experiment)

fig, ax = plt.subplots()
for m in METHODS:
    ax.plot(scores.index, scores[f"WAPE_{m}"], "o-", color=METHOD_COLORS[m], label=m.replace("_", " "))
ax.set(title="How fast does each method learn a new platform?", xlabel="Days since the new platform launched",
       ylabel="WAPE of remaining year failures")
ax.legend()
plt.show()

scores[[f"MAE_{m}" for m in METHODS] + [f"bias_{m}" for m in METHODS]
       + ["hier_80pct_coverage", "hier_vs_platform_only", "hier_vs_fleet_average"]].round(3)

**What to look for.**
1. **Early days:** platform only should be the worst and the most erratic, since a handful of failures in a short window swings it wildly.
2. **Fleet average** never learns. Its error reflects how different the held out platforms really are.
3. **Hierarchical** should sit at or below the better of the two at every cutoff: close to the fleet average when data is scarce, tracking the platform's own data as it accumulates. The two `hier_vs_` columns give the percentage reduction in MAE. Late in the year, platform only can catch up once each cell has many failures; that crossover day is itself a useful answer ("after about N days, the platform's own data is enough").
4. **Coverage** near 80% means the predictive interval can be used for reserves.

Report the results as they come out. If pooling does not help on this fleet, the most likely reason is that platforms differ in ways one shared prior cannot express, such as a component that fails on some platforms and never on others. That is a finding about the fleet, and it points to the extensions in the summary.

### One platform up close
The aggregate hides the mechanism. The chart below follows one launch (day 60) and the held out cell where the fleet average misses the remaining failures by the most 30 days after launch, the case where learning quickly matters most.

In [ ]:
SHOWCASE_LAUNCH = 60
day30 = experiment[(experiment["days"] == 30) & (experiment["launch_day"] == SHOWCASE_LAUNCH)].copy()
day30["fleet_miss"] = (day30["pred_fleet_average"] - day30["actual_remaining"]).abs()
showcase = day30.sort_values("fleet_miss", ascending=False).iloc[0][["held_out", "comp"]]
case = experiment[(experiment["held_out"] == showcase["held_out"]) & (experiment["comp"] == showcase["comp"])
                  & (experiment["launch_day"] == SHOWCASE_LAUNCH)]

fig, ax = plt.subplots()
ax.fill_between(case["days"], case["hier_rate_p10"], case["hier_rate_p90"], color=METHOD_COLORS["hierarchical"],
                alpha=0.2, label="hierarchical 80% interval")
for m in METHODS:
    ax.plot(case["days"], case[f"rate_{m}"], "o-", color=METHOD_COLORS[m], label=m.replace("_", " "))
ax.axhline(case["full_year_rate"].iloc[0], color="black", ls="--", label="full year estimate (hindsight)")
for _, r in case.iterrows():
    ax.annotate(f"{int(r['failures_seen'])} f", (r["days"], r["rate_hierarchical"]), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set(title=f"{showcase['comp']} on {showcase['held_out']}, treated as a new platform",
       xlabel="Days since launch", ylabel="Hazard multiplier")
ax.legend(fontsize=8)
plt.show()

case[["days", "failures_seen", "rate_platform_only", "rate_fleet_average", "rate_hierarchical",
      "actual_remaining", "pred_platform_only", "pred_fleet_average", "pred_hierarchical"]].round(2)

The labels show how many failures had been seen at each cutoff. The hierarchical estimate starts near the fleet center, walks toward the platform's own level as failures accumulate, and its interval narrows. The dashed hindsight line is for orientation only; it uses the whole year.

## 7. What it means for a contract
Translate the showcase platform's estimates **30 days after launch** into a hypothetical three year service contract: the new platform ships 500 tools, and each failure costs the expected corrective cost from Part 3's illustrative cost model.

Expected failures per tool per year = multiplier × risk exposure per tool per year, where exposure per tool comes from the held out platform's own installed base. The hindsight row uses the failures the platform actually had over the full year.

In [ ]:
TOOLS, YEARS, DAYS_SEEN = 500, 3, 30
PART_COST = {"comp1": 8_000, "comp2": 12_000, "comp3": 5_000, "comp4": 9_000}
CORRECTIVE_MEAN = {c: (v * 1.30 + 2_500) * np.exp(0.40 ** 2 / 2) for c, v in PART_COST.items()}  # Part 3 cost model

comp, platform = showcase["comp"], showcase["held_out"]
n_machines = lifetimes.loc[lifetimes["model"] == platform, "machineID"].nunique()
r = case[case["days"] == DAYS_SEEN].iloc[0]
exposure_per_tool_year = (r["exposure_seen"] + r["exposure_remaining"]) / n_machines
actual_per_tool_year = (r["failures_seen"] + r["actual_remaining"]) / n_machines

scenarios = {"platform only": r["rate_platform_only"] * exposure_per_tool_year,
             "fleet average": r["rate_fleet_average"] * exposure_per_tool_year,
             "hierarchical mean": r["rate_hierarchical"] * exposure_per_tool_year,
             "hierarchical P90 (reserve)": r["hier_rate_p90"] * exposure_per_tool_year,
             "hindsight (full year actual)": actual_per_tool_year}

contract = pd.DataFrame({"failures_per_tool_year": scenarios})
contract["contract_failures"] = contract["failures_per_tool_year"] * TOOLS * YEARS
contract["contract_cost"] = contract["contract_failures"] * CORRECTIVE_MEAN[comp]
contract["miss_vs_hindsight"] = contract["contract_cost"] - contract.loc["hindsight (full year actual)", "contract_cost"]

print(f"{comp} on {platform} ({n_machines} machines), {DAYS_SEEN} days after launch: {int(r['failures_seen'])} failures seen, "
      f"${CORRECTIVE_MEAN[comp]:,.0f} per failure")
contract.round(2)

**Reading the contract table.** `miss_vs_hindsight` is how far each estimate available at day 30 would have mispriced the contract compared with what the full year showed. A negative miss is an under reserve that becomes a loss; a positive miss is a price that may lose the deal to a competitor. The hierarchical P90 row is the conservative reserve figure the credible interval supports.

## 8. Prior sensitivity
With only 12 known cells, the hyperprior on α could matter. Rerun the day 60 launch's 30 day forecasts of remaining failures under two alternatives:
1. **flat_log:** every α on the grid equally plausible a priori.
2. **expects_different:** a prior that believes platforms differ a lot (centered on α = 1), which shrinks less.

If the MAE barely moves, the conclusions are driven by the data. If it moves a lot, the choice of prior must be defended in the README.

In [ ]:
sens_rows = []
for prior in ["default", "flat_log", "expects_different"]:
    errs = []
    for held_out in PLATFORMS:
        known, new = lifetimes[lifetimes["model"] != held_out], lifetimes[lifetimes["model"] == held_out]
        baselines = fit_baselines(known)
        w = hyper_posterior(cell_stats(known, baselines), prior=prior)
        launch = OBS_START + pd.Timedelta(days=SHOWCASE_LAUNCH)
        full = cell_stats(window_view(new, launch, OBS_END), baselines).set_index("comp")
        seen = cell_stats(window_view(new, launch, launch + pd.Timedelta(days=30)), baselines).set_index("comp")
        for comp in COMPONENTS:
            y = seen.loc[comp, "failures"] if comp in seen.index else 0
            E = seen.loc[comp, "exposure"] if comp in seen.index else 0.0
            E_rem = max(full.loc[comp, "exposure"] - E, 0.0)
            errs.append(abs(cell_posterior(y, E, w)["mean"] * E_rem - (full.loc[comp, "failures"] - y)))
    sens_rows.append({"prior": prior, "MAE_remaining_failures_at_30_days": np.mean(errs)})

pd.DataFrame(sens_rows).set_index("prior").round(3)

## 9. Save outputs

In [ ]:
experiment.to_csv(DATA_PROCESSED / "new_platform_experiment.csv", index=False)
scores.to_csv(DATA_PROCESSED / "new_platform_scores_by_days.csv")
all_cells.to_csv(DATA_PROCESSED / "platform_multipliers.csv", index=False)
print("Saved: new_platform_experiment.csv, new_platform_scores_by_days.csv, platform_multipliers.csv")

### Summary
1. Measured exposure in **units of cumulative hazard** from the Part 2 Weibull baselines, so platforms with older parts are expected to fail more and are not penalized for it.
2. Built a **hierarchical Gamma Poisson model** whose prior strength is learned from how much existing platforms differ, and computed its posterior exactly on a grid, verified against a hand worked conjugate update and a simulated population.
3. **Held out each platform in turn**, revealed its data over time, and scored platform only, fleet average, and hierarchical forecasts of its remaining year failures, with predictive interval coverage.
4. Translated the 30 day estimates into a **three year contract cost** to show what the choice of method is worth in dollars.
5. Checked **prior sensitivity** so the conclusions can be defended.

### Limitations to state in the README
1. Four platforms is a small population; α is estimated from 12 to 16 cells and would sharpen with a real fleet of dozens of platforms.
2. Remaining exposure is taken from what actually happened rather than projected, which isolates the rate estimate but slightly flatters every method equally.
3. Platforms are assumed exchangeable. A natural extension adds design similarity (shared subsystems, same product family, process node) as a covariate on the prior, so a new platform borrows most from its closest relatives.
4. A platform that is good or bad across all components would motivate a second level (a platform effect shared across components).

## Project wrap up
| Part | Business question | Answer it gives |
|---|---|---|
| 1 | What does the installed base look like, and can we trust the data? | A validated lifetime table with every failure linked and every censored life kept |
| 2 | How do components fail, and what drives it? | Weibull shapes with a failure free period, drivers validated out of sample |
| 3 | What will next quarter cost? | Backtested failure, parts, and cost forecasts for pricing, reserves, and stocking |
| 4 | What about a platform with no history? | A forecast that is useful from the first weeks and converges as data arrives |